<a href="https://colab.research.google.com/github/Anjali2000702/CP-Lab/blob/main/CPL2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from numba import cuda
import numpy as np


# =====================================================
# 1D VECTOR ADDITION KERNEL
# =====================================================
@cuda.jit
def add_vectors(a, b, c):
    i = cuda.grid(1)
    if i < a.size:
        c[i] = a[i] + b[i]


# =====================================================
# 2D MATRIX ADDITION KERNEL
# =====================================================
@cuda.jit
def add_matrices(a, b, c):
    x, y = cuda.grid(2)
    if x < a.shape[0] and y < a.shape[1]:
        c[x, y] = a[x, y] + b[x, y]


def main():

    # =====================================================
    # 1D VECTOR ADDITION
    # =====================================================
    print("---- 1D Vector Addition ----")

    n = 1024

    a = np.arange(n, dtype=np.float32)
    b = np.arange(n, dtype=np.float32)

    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_c = cuda.device_array(n, dtype=np.float32)

    threads_per_block = 256
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

    add_vectors[blocks_per_grid, threads_per_block](d_a, d_b, d_c)

    c = d_c.copy_to_host()

    print("First 10 results:", c[:10])


    # =====================================================
    # 2D MATRIX ADDITION
    # =====================================================
    print("\n---- 2D Matrix Addition ----")

    rows = 32
    cols = 32

    a_2d = np.arange(rows * cols, dtype=np.float32).reshape(rows, cols)
    b_2d = np.arange(rows * cols, dtype=np.float32).reshape(rows, cols)

    d_a_2d = cuda.to_device(a_2d)
    d_b_2d = cuda.to_device(b_2d)
    d_c_2d = cuda.device_array((rows, cols), dtype=np.float32)

    threads_per_block_2d = (16, 16)
    blocks_x = (rows + threads_per_block_2d[0] - 1) // threads_per_block_2d[0]
    blocks_y = (cols + threads_per_block_2d[1] - 1) // threads_per_block_2d[1]

    add_matrices[(blocks_x, blocks_y), threads_per_block_2d](d_a_2d, d_b_2d, d_c_2d)

    c_2d = d_c_2d.copy_to_host()

    print("Top-left 5x5 block:\n", c_2d[:5, :5])


if __name__ == "__main__":
    main()